### ЗАДАЧА: Панель SLA-ребейтов по доставке по паттерну `MVC`

Команда logistics finance разбирает кейсы по SLA-ребейтам: если доставка опоздала,
клиенту или продавцу может быть положена компенсация, а к логистическому партнеру — применен штраф.
Нужно реализовать внутреннюю консольную панель по паттерну `MVC`.

Слои:
- `Model` хранит кейсы и бизнес-правила;
- `View` отвечает только за отображение;
- `Controller` принимает действия и связывает `Model` и `View`.

## Что должно храниться в кейсе

Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `shipment_id` — идентификатор отправления;
- `courier` — служба доставки;
- `promised_days` — обещанный срок доставки;
- `actual_days` — фактический срок доставки;
- `order_value` — стоимость заказа;
- `shipping_fee` — стоимость доставки;
- `penalty_rate` — ставка компенсации за каждый день просрочки;
- `delay_days` — число дней просрочки;
- `requested_rebate` — расчетная сумма ребейта;
- `approved_rebate` — согласованная сумма ребейта;
- `courier_penalty` — штраф, который будет предъявлен логистическому партнеру;
- `status` — статус кейса;
- `coordinator` — сотрудник, который ведет кейс;
- `customer_contacted` — связывались ли с клиентом;
- `decision` — финальное решение.

## Формулы

При создании кейса и после изменения одобренной суммы нужно считать:
- `delay_days = max(actual_days - promised_days, 0)`
- `requested_rebate = min(order_value * penalty_rate * delay_days, shipping_fee + order_value * 0.2)`
- `approved_rebate` при создании равно `0.0`
- `courier_penalty = approved_rebate * 0.7`
- все денежные значения округляются до 2 знаков.

## Статусы

- `new`
- `investigating`
- `customer_contacted`
- `ready_for_approval`
- `approved`
- `rejected`
- `escalated`

## Бизнес-правила

- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить `coordinator` несуществующему кейсу;
- финальные кейсы (`approved`, `rejected`, `escalated`) нельзя менять дальше;
- начать расследование можно только из `new` и только если назначен `coordinator`;
- связаться с клиентом можно только из `investigating`;
- при контакте с клиентом поле `customer_contacted` должно стать `True`, а статус — `customer_contacted`;
- установить `approved_rebate` можно только из `investigating` или `customer_contacted`;
- `approved_rebate` не может быть меньше `0`;
- `approved_rebate` не может быть больше `requested_rebate`;
- после изменения `approved_rebate` нужно пересчитать `courier_penalty`;
- перевод в `ready_for_approval` возможен только из `investigating` или `customer_contacted`;
- перевод в `ready_for_approval` возможен только если `approved_rebate > 0`;
- завершить кейс как `approved` можно только из `ready_for_approval`;
- завершить кейс как `rejected` можно только из `ready_for_approval`, если `approved_rebate == 0`;
- `escalated` можно сделать только из `investigating`, `customer_contacted` или `ready_for_approval`;
- при финальном статусе нужно записывать `decision`.

## Что должен уметь `Model`

Нужно самостоятельно спроектировать модель, но она должна уметь минимум:
- создавать кейс;
- назначать координатора;
- начинать расследование;
- отмечать контакт с клиентом;
- устанавливать `approved_rebate`;
- переводить кейс в `ready_for_approval`;
- завершать кейс как `approved`;
- завершать кейс как `rejected`;
- эскалировать кейс;
- возвращать список кейсов;
- возвращать summary.

## Что должен уметь `View`

Нужно реализовать вывод:
- списка кейсов;
- summary;
- успешных сообщений;
- ошибок.

## Формат строки кейса

Каждый кейс можно вывести строкой такого вида:

`case_id | shipment_id | courier | promised_days | actual_days | delay_days | order_value | shipping_fee | requested_rebate | approved_rebate | courier_penalty | status | coordinator | customer_contacted | decision`

## Что должно быть в summary

Нужно вернуть словарь, в котором есть:
- количество кейсов по статусам;
- `total_requested_rebate` — общая расчетная сумма ребейтов;
- `total_approved_rebate` — общая согласованная сумма;
- `total_courier_penalty` — общая сумма штрафов логистическому партнеру;
- `delayed_shipments` — количество кейсов, где `delay_days > 0`;
- `contacted_cases` — количество кейсов, где `customer_contacted == True`.

## Что нужно сделать в конце

1. Создать модель, view и controller.
2. Загрузить `initial_cases`.
3. Обработать все действия из `actions`.
4. В конце вывести список кейсов и summary.

In [ ]:
initial_cases = [
    ("DR-100", "SHP-9901", "FastBox", 2, 5, 4200.0, 300.0, 0.03),
    ("DR-101", "SHP-9902", "QuickShip", 3, 3, 1800.0, 220.0, 0.02),
]

actions = [
    ("show",),
    ("investigate", "DR-100"),
    ("assign", "DR-100", "Olga"),
    ("investigate", "DR-100"),
    ("contact", "DR-100"),
    ("set_rebate", "DR-100", 180.0),
    ("ready", "DR-100"),
    ("approve", "DR-100", "rebate_sent_to_customer"),
    ("create", "DR-102", "SHP-9903", "CityRun", 1, 4, 2600.0, 180.0, 0.04),
    ("assign", "DR-102", "Max"),
    ("investigate", "DR-102"),
    ("set_rebate", "DR-102", 150.0),
    ("ready", "DR-102"),
    ("escalate", "DR-102", "courier_disputes_delay_window"),
    ("create", "DR-103", "SHP-9904", "ParcelWay", 2, 6, 5200.0, 450.0, 0.03),
    ("assign", "DR-103", "Ina"),
    ("investigate", "DR-103"),
    ("set_rebate", "DR-103", 0.0),
    ("ready", "DR-103"),
    ("reject", "DR-103", "no_customer_refund_required"),
    ("show",),
]

class SLARebateModel:
    def __init__(self):
        self.cases = {}
        self.final_statuses = {'approved', 'rejected', 'escalated'}

    def create_case(self, case_id, shipment_id, courier, promised_days, actual_days,
                   order_value, shipping_fee, penalty_rate):
        if case_id in self.cases:
            raise ValueError(f"Кейс с ID {case_id} уже существует")

        delay_days = max(actual_days - promised_days, 0)
        requested_rebate = min(
            order_value * penalty_rate * delay_days,
            shipping_fee + order_value * 0.2
        )
        requested_rebate = round(requested_rebate, 2)

        self.cases[case_id] = {
            'case_id': case_id,
            'shipment_id': shipment_id,
            'courier': courier,
            'promised_days': promised_days,
            'actual_days': actual_days,
            'delay_days': delay_days,
            'order_value': order_value,
            'shipping_fee': shipping_fee,
            'penalty_rate': penalty_rate,
            'requested_rebate': requested_rebate,
            'approved_rebate': 0.0,
            'courier_penalty': 0.0,
            'status': 'new',
            'coordinator': None,
            'customer_contacted': False,
            'decision': None
        }

    def assign_coordinator(self, case_id, coordinator):
        case = self._get_case(case_id)
        if case['status'] in self.final_statuses:
            raise ValueError("Финальный кейс нельзя изменить")
        case['coordinator'] = coordinator

    def investigate(self, case_id):
        case = self._get_case(case_id)
        if case['status'] in self.final_statuses:
            raise ValueError("Финальный кейс нельзя изменить")
        if case['status'] != 'new':
            raise ValueError("Расследование можно начать только из статуса 'new'")
        if not case['coordinator']:
            raise ValueError("Для начала расследования нужно назначить координатора")
        case['status'] = 'investigating'

    def contact_customer(self, case_id):
        case = self._get_case(case_id)
        if case['status'] in self.final_statuses:
            raise ValueError("Финальный кейс нельзя изменить")
        if case['status'] != 'investigating':
            raise ValueError("Контакт с клиентом возможен только из статуса 'investigating'")
        case['customer_contacted'] = True
        case['status'] = 'customer_contacted'

    def set_approved_rebate(self, case_id, amount):
        case = self._get_case(case_id)
        if case['status'] in self.final_statuses:
            raise ValueError("Финальный кейс нельзя изменить")
        if case['status'] not in ['investigating', 'customer_contacted']:
            raise ValueError("Установку approved_rebate можно делать только из статусов 'investigating' или 'customer_contacted'")
        if amount < 0:
            raise ValueError("approved_rebate не может быть меньше 0")
        if amount > case['requested_rebate']:
            raise ValueError("approved_rebate не может быть больше requested_rebate")

        case['approved_rebate'] = round(amount, 2)
        case['courier_penalty'] = round(case['approved_rebate'] * 0.7, 2)

    def ready_for_approval(self, case_id):
        case = self._get_case(case_id)
        if case['status'] in self.final_statuses:
            raise ValueError("Финальный кейс нельзя изменить")
        if case['status'] not in ['investigating', 'customer_contacted']:
            raise ValueError("Перевод в ready_for_approval возможен только из статусов 'investigating' или 'customer_contacted'")
        if case['approved_rebate'] <= 0:
            raise ValueError("Перевод в ready_for_approval возможен только если approved_rebate > 0")
        case['status'] = 'ready_for_approval'

    def approve(self, case_id, decision):
        case = self._get_case(case_id)
        if case['status'] != 'ready_for_approval':
            raise ValueError("Завершение как approved возможно только из статуса 'ready_for_approval'")
        case['status'] = 'approved'
        case['decision'] = decision

    def reject(self, case_id, decision):
        case = self._get_case(case_id)
        if case['status'] != 'ready_for_approval':
            raise ValueError("Завершение как rejected возможно только из статуса 'ready_for_approval'")
        if case['approved_rebate'] != 0:
            raise ValueError("rejected возможен только если approved_rebate == 0")
        case['status'] = 'rejected'
        case['decision'] = decision

    def escalate(self, case_id, decision):
        case = self._get_case(case_id)
        if case['status'] in self.final_statuses:
            raise ValueError("Финальный кейс нельзя изменить")
        if case['status'] not in ['investigating', 'customer_contacted', 'ready_for_approval']:
            raise ValueError("Эскалация возможна только из статусов 'investigating', 'customer_contacted' или 'ready_for_approval'")
        case['status'] = 'escalated'
        case['decision'] = decision

    def get_cases(self):
        return list(self.cases.values())

    def get_summary(self):
        summary = {
            'status_counts': {},
            'total_requested_rebate': 0.0,
            'total_approved_rebate': 0.0,
            'total_courier_penalty': 0.0,
            'delayed_shipments': 0,
            'contacted_cases': 0
        }

        for case in self.cases.values():
            status = case['status']
            summary['status_counts'][status] = summary['status_counts'].get(status, 0) + 1
            summary['total_requested_rebate'] += case['requested_rebate']
            summary['total_approved_rebate'] += case['approved_rebate']
            summary['total_courier_penalty'] += case['courier_penalty']

            if case['delay_days'] > 0:
                summary['delayed_shipments'] += 1
            if case['customer_contacted']:
                summary['contacted_cases'] += 1

        summary['total_requested_rebate'] = round(summary['total_requested_rebate'], 2)
        summary['total_approved_rebate'] = round(summary['total_approved_rebate'], 2)
        summary['total_courier_penalty'] = round(summary['total_courier_penalty'], 2)

        return summary

    def _get_case(self, case_id):
        if case_id not in self.cases:
            raise ValueError(f"Кейс с ID {case_id} не найден")
        return self.cases[case_id]
        
class SLARebateView:
    @staticmethod
    def display_cases(cases):
        print("Список кейсов:")
        header = "case_id | shipment_id | courier | promised_days | actual_days | delay_days | order_value | shipping_fee | requested_rebate | approved_rebate | courier_penalty | status | coordinator | customer_contacted | decision"
        print(header)

        for case in cases:
            row = (
                f"{case['case_id']} | {case['shipment_id']} | {case['courier']} | "
                f"{case['promised_days']} | {case['actual_days']} | {case['delay_days']} | "
                f"{case['order_value']} | {case['shipping_fee']} | {case['requested_rebate']} | "
                f"{case['approved_rebate']} | {case['courier_penalty']} | {case['status']} | "
                f"{case['coordinator'] if case['coordinator'] else 'None'} | "
                f"{case['customer_contacted']} | "
                f"{case['decision'] if case['decision'] else 'None'}"
            )
            print(row)
        print()

    @staticmethod
    def display_summary(summary):
        print("Summary:")
        print(f"Количество кейсов по статусам: {summary['status_counts']}")
        print(f"Общая расчётная сумма ребейтов: {summary['total_requested_rebate']}")
        print(f"Общая согласованная сумма: {summary['total_approved_rebate']}")
        print(f"Общая сумма штрафов логистическому партнёру: {summary['total_courier_penalty']}")
        print(f"Количество задержанных отправлений: {summary['delayed_shipments']}")
        print(f"Количество кейсов, где связались с клиентом: {summary['contacted_cases']}")
        print()

    @staticmethod
    def display_success(message):
        print(f"[SUCCESS] {message}")

    @staticmethod
    def display_error(message):
        print(f"[ERROR] {message}")
        
class SLARebateController:
    def __init__(self, model, view):
        self.model = model
        self.view = view

    def handle_action(self, action):
        try:
            command = action[0]

            if command == "show":
                cases = self.model.get_cases()
                self.view.display_cases(cases)

            elif command == "create":
                _, case_id, shipment_id, courier, promised_days, actual_days, order_value, shipping_fee, penalty_rate = action
                self.model.create_case(case_id, shipment_id, courier, promised_days, actual_days,
                                           order_value, shipping_fee, penalty_rate)
                self.view.display_success(f"Кейс {case_id} создан")

            elif command == "assign":
                _, case_id, coordinator = action
                self.model.assign_coordinator(case_id, coordinator)
                self.view.display_success(f"Координатор {coordinator} назначен для кейса {case_id}")

            elif command == "investigate":
                _, case_id = action
                self.model.investigate(case_id)
                self.view.display_success(f"Расследование начато для кейса {case_id}")

            elif command == "contact":
                _, case_id = action
                self.model.contact_customer(case_id)
                self.view.display_success(f"Контакт с клиентом отмечен для кейса {case_id}")

            elif command == "set_rebate":
                _, case_id, amount = action
                self.model.set_approved_rebate(case_id, float(amount))
                self.view.display_success(f"approved_rebate установлен в {amount} для кейса {case_id}")

            elif command == "ready":
                _, case_id = action
                self.model.ready_for_approval(case_id)
                self.view.display_success(f"Кейс {case_id} переведён в статус ready_for_approval")

            elif command == "approve":
                _, case_id, decision = action
                self.model.approve(case_id, decision)
                self.view.display_success(f"Кейс {case_id} утверждён с решением: {decision}")

            elif command == "reject":
                _, case_id, decision = action
                self.model.reject(case_id, decision)
                self.view.display_success(f"Кейс {case_id} отклонён с решением: {decision}")

            elif command == "escalate":
                _, case_id, decision = action
                self.model.escalate(case_id, decision)
                self.view.display_success(f"Кейс {case_id} эскалатирован с решением: {decision}")

            else:
                self.view.display_error(f"Неизвестная команда: {command}")

        except Exception as e:
            self.view.display_error(str(e))

    def show_summary(self):
        summary = self.model.get_summary()
        self.view.display_summary(summary)

model = SLARebateModel()
view = SLARebateView()
controller = SLARebateController(model, view)

for case_data in initial_cases:
    controller.handle_action(("create",) + case_data)

for action in actions:
    controller.handle_action(action)

controller.show_summary()






        
